# Handoff: analytic partial derivatives of the Bolin sensitivity Be

**Status: the derivation is DONE and validated.** This notebook replaces the earlier
handoff, which asked for the derivation and, in doing so, stated three things that turned
out to be wrong. They are listed below because two of them were expensive.

The remaining task is the **mocsy (Fortran) port**.

## Deliverables that now exist

| file | what it is |
|---|---|
| `buffderiv.R` | the four analytic derivatives, dBe/dT, dBe/dS, dBe/dCT, dBe/dAT |
| `dlnK.R` | analytic dln(K)/dT and dln(K)/dS for every constant (the only hand-coded part) |
| `buffsun.R` | Be and Rf (renamed from `bufforr.R`, for Sundquist et al. 1979) |
| `buffer.R`, `buffesm.R`, `buffzwg.R` | corrected, see below |
| `validate_buffderiv.R`, `sweep.R`, `crossvalidate_buff.R` | the test suite |
| `reference_values.csv` | 10 reference points; **the target for the Fortran port** |

## Corrections to the previous handoff (read these)

**1. The alkalinity equation was missing the fluoride term.** SolveSAPHE, which `carb()`
inverts, carries `-[HF]` explicitly. It is worth up to 5e-7 of AT, which displaces h by
~6e-7 relative. Omitting it meant differentiating an alkalinity that seacarb does not use.
The correct non-carbonate buffer term is

$$ w = \underbrace{\frac{1}{\sigma} + \frac{S_T K_S'}{(K_S'+h)^2}}_{\text{= 1 to } 2\times10^{-8}} + \frac{K_w}{h^2} + \frac{B_T K_B}{(K_B+h)^2} + w_{Si} + w_P + \underbrace{\frac{F_T K_F'}{(K_F'+h)^2}}_{\text{was missing}} + w_N + w_S $$

with $\sigma = 1 + S_T/K_S$, $K_S' = K_S + S_T$, and $K_F' = K_F\,\sigma$. **Kf must be put on
the same pH scale as h**; using the free-scale Kf with a total-scale h is wrong by 28%.
The old `1` at the front is exactly the proton+sulfate pair, so sulfate needs no change.

**2. Silicate is monoprotic on the `carb()` path, not diprotic.** `carb()` calls
`calculate_carb(fullresult=FALSE)`, whose alkalinity has `siooh3` only. Only `carbfull()`
(`fullresult=TRUE`) is diprotic. The diprotic $w_{Si}$ collapses to the monoprotic form
*exactly* at `K2si = 0`, so the fix is to tie K2si to the solver. The old handoff asserted
diprotic unconditionally.

**3. Nutrients are NOT negligible.** The old handoff guessed that X reduces to borate+water.
It does not: Be at the polar reference point is 7.737 with Pt/Sit and 7.868 without, a 1.7%
difference. Phosphate and silicate stay in.

## Two seacarb traps. Both silent. Both bite polar CMIP6 cells.

- **`k1k2="x"`** (K1.R): `is_outrange <- T>35 | T<2 | S<19 | S>43`, then switches to
  Waters et al. (2014). Below 2 degC you silently get a different formulation.
- **`kf="x"`** (Kf.R): `is_outrange <- T>33 | T<10 | S<10 | S>40`, Perez & Fraga in range,
  Dickson & Riley outside. This changes kSWS2total and hence Kw, Ksi, K1p, K2p, K3p by
  **0.5 to 0.9 percent**.

Always pass `k1k2="l"`, `kf="dg"`, `ks="d"`, `b="u74"`, `pHscale="T"` explicitly.
`buffderiv()` hard-stops on anything else. The old handoff's harness used the defaults.

## The method (unchanged, and it works)

h is fixed implicitly by the alkalinity constraint

$$ F(h;T,S,C_T,A_T) = A_c(h,C_T) + A_{nc}(h) - A_T = 0 $$

and Be is the *explicit* algebraic function $G(h,C_T,T,S)$ that `buffsun()` evaluates:

$$ B_e = 1 + \frac{c}{s} + \frac{Xb - 4c^2}{s\,(b+4c+X)}, \qquad X = h\,w $$

So for any Y in {T, S, CT, AT}, the implicit function theorem gives

$$ \frac{dB_e}{dY} = G_Y + G_h\left(-\frac{F_Y}{F_h}\right) $$

Everything closed form. The only nonlinear solve is the single `carb()` call for h.
`buffderiv()` then applies 2-3 Newton steps to h using $F_h$ (which it needs anyway),
because `carb()` converges h only to ~1e-10 relative and Be is steep in h.

In [ ]:
source("setup.R")   # library(seacarb) + the buffer functions from ../R
S0 <- 35; T0 <- 20; CT0 <- 2000e-6; AT0 <- 2300e-6
P0 <- 0.2e-6; Si0 <- 3e-6

d <- buffderiv(flag = 15, var1 = AT0, var2 = CT0, S = S0, T = T0,
               Pt = P0, Sit = Si0,
               k1k2 = "l", kf = "dg", ks = "d", pHscale = "T", b = "u74")
d

## The one diagnostic that matters

`alk_residual` = Ac(h) + Anc(h) - ALK. If our Anc is not *exactly* the alkalinity seacarb
solved, we are differentiating the wrong function and every derivative is quietly wrong.

Evaluate it at `carb()`'s **unpolished** h (`npolish = 0`). That is the honest test: with
the Newton polish on, the residual is driven to zero by construction and tells you nothing.

- missing the [HF] term: **5.2e-7**
- with [HF]: **9.7e-11**  (this is just `carb()`'s own convergence tolerance)

In [ ]:
buffderiv(15, AT0, CT0, S = S0, T = T0, Pt = P0, Sit = Si0,
          k1k2 = "l", kf = "dg", ks = "d", pHscale = "T", b = "u74",
          npolish = 0)$alk_residual / AT0

## Validation status

| check | worst error |
|---|---|
| `dlnK.R` dK/dT, dK/dS vs FD of seacarb's own K functions | 9.6e-7 % |
| alkalinity residual at `carb()`'s unpolished h | 9.7e-11 |
| alkalinity residual after Newton polish | 4.1e-16 |
| dBe/dY vs FD through `carb()`, 310 points, both solver paths | **1.1e-6 %** |
| Be across `buffsun`/`buffzwg`/`buffesm`/`buffer`/`buffderiv` | 2.5e-10 |
| Rf across `buffsun`/`buffzwg`/`buffesm`/`buffer` | 2.7e-15 |

The 1.1e-6 % is the double-precision FD noise floor, not analytic error: the same
derivation in 50-digit arithmetic, cross-checked against an independent sympy implicit
differentiation, agrees to 1.1e-11 %.

## Remaining task: the mocsy Fortran port

Everything after the `carb()` call is pure algebra, so the Fortran routine should take
**pH as an input** (from mocsy's `vars()`) and do **no solve**. Signature along the lines of

```fortran
SUBROUTINE dbe(Be, dBe_dt, dBe_ds, dBe_ddic, dBe_dalk, &
               temp, sal, dic, alk, pt, sit, ph, N)
```

Notes for whoever does it:

- Fortran is **case-insensitive**, so `G_s` and `G_S` collide. Use `Gs`, `Gsal`, `Gtem`,
  `Gh`, `Gct`, `Gx`.
- mocsy's constants must be checked against seacarb's, one at a time, before trusting any
  derivative. In particular confirm mocsy's `bor` is Uppstrom `0.1284*S*1e-3/10.811`.
- **`reference_values.csv` is the target.** Match it, then re-run the FD check in Fortran
  independently. Do not assume the port is right because it compiles.
- The Newton polish is 6 lines and needs `F_h`, which is already computed. Keep it.